# Notebook 2: Model Training, Registry & Environment Promotion

Train an XGBoost PD model with hyperparameter tuning, evaluate comprehensively (including SHAP and calibration), register in DEV, and promote through STAGING to PROD.

**Pipeline mapping**: Replaces SageMaker HPO Step + Train Step + Evaluation Step + Condition Step + Model Registry + Terraform-based env promotion

**Cost/performance advantages over SageMaker:**
- Training runs on warehouse compute with auto-suspend -- no always-on SageMaker notebook instance ($0.0464/hr for ml.t3.medium)
- No S3 artifact storage costs -- model artifacts stored natively in Snowflake
- HPO runs in-notebook -- no separate SageMaker Tuning Job (eliminates job orchestration overhead)
- Environment promotion is Python API calls, not Terraform deployments per environment
- Model Registry is a schema-level object with RBAC -- no separate approval workflows to configure

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, roc_curve, precision_recall_curve,
                             confusion_matrix, classification_report,
                             brier_score_loss, average_precision_score)
import matplotlib.pyplot as plt
import seaborn as sns
from snowflake.snowpark.context import get_active_session

session = get_active_session()
session.use_database('PD_DEMO_DEV')
session.use_schema('ML')

df = session.table('PD_DEMO_DEV.ML.TRAINING_DATA').to_pandas()
print(f"Loaded training data: {df.shape}")

## 1. Data Preparation & Train/Test Split

In [ ]:
# 20 model input features: 17 numeric (bureau + application) + 3 one-hot (origination channel)
# Preprocessing: missing value imputation handled here; in production, can also be done
# as SQL transforms in the Feature Store Dynamic Table (Snowflake-native, no pandas needed)
feature_cols = [
    'NUM_CREDIT_PRODUCTS', 'NUM_INACTIVE_ACCOUNTS', 'NUM_CREDIT_SEARCHES_L6M',
    'TOTAL_OUTSTANDING_BALANCE', 'MAX_DELINQUENCY_DAYS', 'CREDIT_UTILISATION_RATIO',
    'MONTHS_SINCE_OLDEST_ACCOUNT', 'NUM_DEFAULTS_L3Y', 'NUM_CCJS', 'CREDIT_SCORE',
    'NUM_OPEN_ACCOUNTS', 'TOTAL_CREDIT_LIMIT', 'NUM_MISSED_PAYMENTS_L12M',
    'DEBT_TO_INCOME_RATIO', 'MONTHS_SINCE_LAST_DEFAULT', 'NUM_HARD_SEARCHES_L3M',
    'APPLICANT_AGE_YEARS',
    'CHANNEL_DIRECT', 'CHANNEL_GOOGLE', 'CHANNEL_META'
]
target_col = 'DEFAULT_90DPM'

# Missing value imputation -- median for numeric features
X = df[feature_cols].fillna(df[feature_cols].median())
y = df[target_col]

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print(f"Train: {X_train.shape[0]:,} | Val: {X_val.shape[0]:,} | Test: {X_test.shape[0]:,}")
print(f"Features: {X_train.shape[1]} model input features")
print(f"Train default rate: {y_train.mean():.2%} | Val: {y_val.mean():.2%} | Test: {y_test.mean():.2%}")

pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"Positive class weight: {pos_weight:.1f}")

## 2. Baseline Model

In [ ]:
# Training runs on Snowflake warehouse compute -- auto-suspends when idle
# SageMaker: you pay for a notebook instance 24/7 unless you manually stop it
# Snowflake: warehouse suspends after 60s of inactivity, resumes in <1s
baseline_model = xgb.XGBClassifier(
    n_estimators=100, max_depth=5, learning_rate=0.1,
    scale_pos_weight=pos_weight, eval_metric='auc',
    random_state=42, use_label_encoder=False
)
baseline_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

y_pred_proba = baseline_model.predict_proba(X_test)[:, 1]
baseline_auc = roc_auc_score(y_test, y_pred_proba)
baseline_gini = 2 * baseline_auc - 1

print(f"Baseline AUC: {baseline_auc:.4f}")
print(f"Baseline Gini: {baseline_gini:.4f}")

## 3. Hyperparameter Tuning

Replaces SageMaker HyperParameter Tuning Step. Runs Bayesian optimisation in-notebook -- no separate SageMaker Tuning Job needed (eliminates job scheduling overhead and reduces cost by using existing warehouse compute). For larger-scale HPO, Snowflake also provides a native `Tuner` API that distributes trials across compute nodes.

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'scale_pos_weight': pos_weight,
        'eval_metric': 'auc',
        'random_state': 42,
        'use_label_encoder': False,
    }
    model = xgb.XGBClassifier(**params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    y_pred = model.predict_proba(X_val)[:, 1]
    return roc_auc_score(y_val, y_pred)

study = optuna.create_study(direction='maximize', study_name='pd_model_hpo')
study.optimize(objective, n_trials=30, show_progress_bar=True)

print(f"\nBest AUC (validation): {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

In [ ]:
# =============================================================================
# TRAINING COST COMPARISON (honest assessment)
# Reported SageMaker pipeline: ~2.5 hr @ $1/hr = $2.50/run
# Snowflake: MEDIUM warehouse = 4 credits/hr @ ~$3/credit = ~$12/hr
# Auto-suspend after 60s means you pay only for active compute (~15-20 min)
# =============================================================================
print("""
=== Training Pipeline Cost: SageMaker vs Snowflake ===

                        SageMaker               Snowflake
---------------------------------------------------------------
Instance type           ml instance ($1/hr)     MEDIUM WH (4 credits/hr @ ~$3 = ~$12/hr)
Runtime                 ~2.5 hours              ~15-20 min active (auto-suspend between steps)
Auto-suspend            Manual stop required    60s idle timeout (built-in)
Cost per training run   ~$2.50                  ~$4-5

Honest take:
  - SageMaker training is cheaper per run ($2.50 vs ~$4-5)
  - Snowflake warehouse costs more per hour but runs for much less time
  - The ~$2 premium per run is the trade-off for auto-suspend + no S3 staging

Where the real savings are (not in training compute):
  - Staging endpoint: $92/month always-on (SageMaker) vs ~$5/month (Snowflake)
  - Prod endpoint (bursty): $92/month vs ~$43/month (8hr/day on SPCS CPU_X64_XS)
  - Eliminated services: S3, CloudWatch, EventBridge, notebook instance
  - Engineering time: no Terraform, no IAM, no ECR per environment
  - Annual saving: ~$4,300-7,400 per model pipeline (see Notebook 3 for full breakdown)
""")

## 4. Train Final Model with Best Hyperparameters

In [ ]:
best_params = study.best_params
best_params['scale_pos_weight'] = pos_weight
best_params['eval_metric'] = 'auc'
best_params['random_state'] = 42
best_params['use_label_encoder'] = False

best_model = xgb.XGBClassifier(**best_params)
best_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

y_pred_proba = best_model.predict_proba(X_test)[:, 1]
y_pred = best_model.predict(X_test)

auc_score = roc_auc_score(y_test, y_pred_proba)
gini = 2 * auc_score - 1
brier = brier_score_loss(y_test, y_pred_proba)
avg_precision = average_precision_score(y_test, y_pred_proba)

fpr_arr, tpr_arr, _ = roc_curve(y_test, y_pred_proba)
ks_stat = max(tpr_arr - fpr_arr)

print(f"=== Final Model Performance (Test Set) ===")
print(f"AUC:              {auc_score:.4f}  (baseline: {baseline_auc:.4f}, delta: {auc_score - baseline_auc:+.4f})")
print(f"Gini:             {gini:.4f}")
print(f"KS Statistic:     {ks_stat:.4f}")
print(f"Brier Score:      {brier:.4f}  (lower is better)")
print(f"Avg Precision:    {avg_precision:.4f}")
print(f"\n{classification_report(y_test, y_pred, target_names=['Non-Default', 'Default'])}")

## 5. Comprehensive Evaluation Suite

ROC/PR curves, SHAP explainability, calibration plot, and confusion matrix.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
axes[0, 0].plot(fpr, tpr, 'b-', lw=2, label=f'AUC = {auc_score:.4f}')
axes[0, 0].plot([0, 1], [0, 1], 'k--', lw=1)
axes[0, 0].set_xlabel('False Positive Rate')
axes[0, 0].set_ylabel('True Positive Rate')
axes[0, 0].set_title('ROC Curve')
axes[0, 0].legend()

precision, recall, _ = precision_recall_curve(y_test, y_pred_proba)
axes[0, 1].plot(recall, precision, 'r-', lw=2, label=f'AP = {avg_precision:.4f}')
axes[0, 1].set_xlabel('Recall')
axes[0, 1].set_ylabel('Precision')
axes[0, 1].set_title('Precision-Recall Curve')
axes[0, 1].legend()

from sklearn.calibration import calibration_curve
prob_true, prob_pred = calibration_curve(y_test, y_pred_proba, n_bins=10)
axes[1, 0].plot(prob_pred, prob_true, 'go-', lw=2, label='Model')
axes[1, 0].plot([0, 1], [0, 1], 'k--', lw=1, label='Perfectly calibrated')
axes[1, 0].set_xlabel('Mean Predicted Probability')
axes[1, 0].set_ylabel('Fraction of Positives')
axes[1, 0].set_title(f'Calibration Plot (Brier={brier:.4f})')
axes[1, 0].legend()

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1, 1],
            xticklabels=['Non-Default', 'Default'], yticklabels=['Non-Default', 'Default'])
axes[1, 1].set_xlabel('Predicted')
axes[1, 1].set_ylabel('Actual')
axes[1, 1].set_title('Confusion Matrix')

plt.tight_layout()
plt.show()

In [ ]:
# SHAP explainability: critical for regulated BNPL lending
# Adverse action notices require feature-level explanations for declined applications
# Snowflake also provides built-in explain() on registered models (warehouse inference)
# SageMaker has no native equivalent -- you'd build this as a custom Processing Step
import shap

explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test.iloc[:500])

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

plt.sca(axes[0])
shap.summary_plot(shap_values, X_test.iloc[:500], plot_type="bar", show=False)
axes[0].set_title('Feature Importance (Mean |SHAP|)')

plt.sca(axes[1])
shap.summary_plot(shap_values, X_test.iloc[:500], show=False)
axes[1].set_title('SHAP Value Distribution')

plt.tight_layout()
plt.show()

## 6. Conditional Registration & Model Registry

Replaces SageMaker Condition Step + Model Registry. Only register if the model meets the performance threshold.

In [ ]:
# =============================================================================
# MODEL REGISTRY: Replaces SageMaker Model Registry + S3 artifact storage
# - Model is a first-class Snowflake schema object with RBAC
# - No S3 bucket for model artifacts -- stored natively in Snowflake
# - target_platforms enables BOTH SQL (warehouse) and REST (SPCS) inference
#   from a single registration -- no separate model packaging or ECR images
# - Metrics stored as model metadata -- no separate model card system
# - Conditional registration mirrors SageMaker Condition Step
# =============================================================================
from snowflake.ml.registry import Registry

AUC_THRESHOLD = 0.70

if auc_score < AUC_THRESHOLD:
    raise ValueError(f"Model AUC {auc_score:.4f} below threshold {AUC_THRESHOLD}. Not registering.")

print(f"AUC {auc_score:.4f} >= threshold {AUC_THRESHOLD}. Proceeding with registration.")

dev_reg = Registry(session=session, database_name="PD_DEMO_DEV", schema_name="REGISTRY")

sample_input = X_test.head(10)

mv = dev_reg.log_model(
    best_model,
    model_name="PD_XGBOOST",
    version_name="V1",
    sample_input_data=sample_input,
    conda_dependencies=["xgboost"],
    target_platforms=["WAREHOUSE", "SNOWPARK_CONTAINER_SERVICES"],
    comment=f"XGBoost PD model - 90 DPM target | AUC={auc_score:.4f} Gini={gini:.4f} KS={ks_stat:.4f}"
)

mv.set_metric("auc", auc_score)
mv.set_metric("gini", gini)
mv.set_metric("ks_statistic", ks_stat)
mv.set_metric("brier_score", brier)
mv.set_metric("avg_precision", avg_precision)

print(f"\nModel registered in DEV: {mv.model_name} v{mv.version_name}")
print(f"Metrics: {mv.show_metrics()}")

In [ ]:
result = mv.run(X_test.head(5), function_name="predict_proba")
print("DEV batch inference test (first 5 rows):")
result

## 7. Environment Promotion: DEV -> STAGING -> PROD

This is the real-life ML Ops workflow. In SageMaker, each promotion requires Terraform to spin up separate endpoints per environment + manual approval in the SM Registry. In Snowflake, it is Python API calls governed by RBAC.

**Cost advantage**: No Terraform plan/apply cycle per environment. No separate endpoint provisioning costs until you actually deploy a service. Model promotion is a metadata operation, not an infrastructure deployment.

In [ ]:
# =============================================================================
# STEP 1: Promote DEV -> STAGING
# In production, this step requires the PD_MLOPS role (RBAC enforced)
# Cost: no Terraform deployment -- just a Python API call (seconds, not minutes)
# =============================================================================
print("=" * 60)
print("STEP 1: Promote DEV -> STAGING")
print("In production, this requires the PD_MLOPS role")
print("=" * 60)

dev_model = dev_reg.get_model("PD_XGBOOST").version("V1")
loaded_model = dev_model.load()
dev_metrics = dev_model.show_metrics()

staging_reg = Registry(session=session, database_name="PD_DEMO_STAGING", schema_name="REGISTRY")

staging_mv = staging_reg.log_model(
    loaded_model,
    model_name="PD_XGBOOST",
    version_name="V1",
    sample_input_data=sample_input,
    conda_dependencies=["xgboost"],
    target_platforms=["WAREHOUSE", "SNOWPARK_CONTAINER_SERVICES"],
    comment="Promoted from DEV - pending validation"
)

for metric_name, metric_val in dev_metrics.items():
    staging_mv.set_metric(metric_name, metric_val)

print(f"Model promoted to STAGING: {staging_mv.model_name} v{staging_mv.version_name}")

In [ ]:
print("=" * 60)
print("STEP 2: Validate in STAGING")
print("=" * 60)

staging_preds = staging_mv.run(X_test, function_name="predict_proba")
staging_proba = staging_preds.iloc[:, -1].values
staging_auc = roc_auc_score(y_test, staging_proba)

tolerance = 0.001
auc_match = abs(staging_auc - auc_score) < tolerance

print(f"DEV AUC:      {auc_score:.6f}")
print(f"STAGING AUC:  {staging_auc:.6f}")
print(f"Delta:        {abs(staging_auc - auc_score):.6f}")
print(f"Validation:   {'PASSED' if auc_match else 'FAILED'} (tolerance={tolerance})")

if not auc_match:
    raise ValueError("Staging validation failed. Model behaviour differs from DEV.")

In [ ]:
# =============================================================================
# STEP 3: Promote STAGING -> PROD
# In SageMaker: Terraform apply -> ECR build -> endpoint deploy (10-15 min per env)
# In Snowflake: one log_model() call (seconds). Endpoint deployed separately in NB03.
# =============================================================================
print("=" * 60)
print("STEP 3: Promote STAGING -> PROD")
print("=" * 60)

prod_reg = Registry(session=session, database_name="PD_DEMO_PROD", schema_name="REGISTRY")

prod_mv = prod_reg.log_model(
    loaded_model,
    model_name="PD_XGBOOST",
    version_name="V1",
    sample_input_data=sample_input,
    conda_dependencies=["xgboost"],
    target_platforms=["WAREHOUSE", "SNOWPARK_CONTAINER_SERVICES"],
    comment=f"Production release | AUC={auc_score:.4f} Gini={gini:.4f} | Validated in STAGING"
)

for metric_name, metric_val in dev_metrics.items():
    prod_mv.set_metric(metric_name, metric_val)

print(f"Model promoted to PROD: {prod_mv.model_name} v{prod_mv.version_name}")

In [ ]:
print("=" * 60)
print("MODEL REGISTRY STATE ACROSS ALL ENVIRONMENTS")
print("=" * 60)

for env in ['PD_DEMO_DEV', 'PD_DEMO_STAGING', 'PD_DEMO_PROD']:
    models = session.sql(f"SHOW MODELS IN SCHEMA {env}.REGISTRY").to_pandas()
    print(f"\n{env}.REGISTRY:")
    if len(models) > 0:
        print(f"  Models: {models['name'].tolist()}")
    else:
        print("  (empty)")

---
## Summary

| What we did | SageMaker equivalent | Snowflake approach |
|---|---|---|
| Baseline + HPO (30 trials) | SageMaker HPO Step | Optuna in Notebook (or native Tuner) |
| Full evaluation (ROC, SHAP, calibration) | SageMaker Evaluation Step | In-notebook, metrics stored with model |
| Conditional registration | SageMaker Condition Step | Python threshold check before `log_model` |
| DEV -> STAGING -> PROD promotion | Terraform per env + SM approval | `log_model` across registries, RBAC-gated |
| Staging validation | Manual endpoint testing | Automated metric comparison |

**Next**: Notebook 3 -- Model Serving, Real-Time Inference, and Taktile Integration